In [1]:
import polars as pl
from datetime import timedelta, date, datetime

# Анализ таргета

In [2]:
from src.config import RAW_DATA_DIR

events_path = RAW_DATA_DIR / "dataset/small/marketplace/events"
items_path = "D:/HSE_repos/dreamteam-recsys/t_ecd_data/dataset/small/marketplace/items.pq"

2025-11-17 12:35:02.333 | INFO     | src.config:<module>:11 - PROJ_ROOT path is: D:\HSE_repos\dreamteam-recsys


In [3]:
events_df = pl.scan_parquet(events_path)

In [4]:
events_df.collect_schema()

Schema([('timestamp', Duration(time_unit='us')),
        ('user_id', UInt64),
        ('item_id', String),
        ('subdomain', String),
        ('action_type', String),
        ('os', String),
        ('day', Int32)])

In [5]:
events_df.head().collect(engine="streaming")

timestamp,user_id,item_id,subdomain,action_type,os,day
duration[μs],u64,str,str,str,str,i32
1082d 89901µs,59328774,"""nfmcg_14777696""","""u2i""","""view""","""android""",1082
1082d 163483µs,66295302,"""nfmcg_26098539""","""catalog""","""view""","""android""",1082
1082d 864625µs,37542104,"""nfmcg_10840247""","""u2i""","""view""","""android""",1082
1082d 889192µs,35193548,"""nfmcg_8040572""","""catalog""","""view""","""android""",1082
1082d 936008µs,27256137,"""nfmcg_6770239""","""catalog""","""view""","""android""",1082


In [6]:
actions_count = events_df.group_by("action_type").agg(pl.len()).collect(engine="streaming")

Создадим также уменьшенный по масштабу таргет: логарифмированный и квадратичный. Идея: постараться естественным (зависимым от статистики) образом создать таргет, при этом не слишком большим по значению, т.к. огромные редкие таргеты могут заставить модели переобучаться и/или искажать оценки ее качества.

Однако гипотезу нужно будет проверить на практике, потому на текущий момент предлагается 3 варианта: стандартный таргет, логарифмированный и таргет в корне:

In [7]:
actions_count = actions_count.with_columns(
    (pl.col("len").sum() / pl.col("len")).alias("target"),
    (pl.col("len").sum() / pl.col("len")).log1p().alias("log_target"),
    (pl.col("len").sum() / pl.col("len")).sqrt().alias("sqrt_target"),
    
).drop("len")
actions_count

action_type,target,log_target,sqrt_target
str,f64,f64,f64
"""clickout""",268.714913,5.597366,16.392526
"""click""",34.582149,3.571844,5.880659
"""like""",3121.341812,8.046339,55.86897
"""view""",1.034082,0.710044,1.016898


In [8]:
events_df = events_df.join(actions_count.lazy(), on="action_type").with_columns([
    (pl.col("action_type") == "view").cast(pl.Int8).alias("target_view"),
    (pl.col("action_type") == "clickout").cast(pl.Int8).alias("target_clickout"),
    (pl.col("action_type") == "like").cast(pl.Int8).alias("target_like"),
    (pl.col("action_type") == "click").cast(pl.Int8).alias("target_click"),
])

In [9]:
events_df.head().collect(engine="streaming")

timestamp,user_id,item_id,subdomain,action_type,os,day,target,log_target,sqrt_target,target_view,target_clickout,target_like,target_click
duration[μs],u64,str,str,str,str,i32,f64,f64,f64,i8,i8,i8,i8
1082d 13h 55m 8s 952090µs,24163521,"""nfmcg_12636714""","""u2i""","""click""","""ios""",1082,34.582149,3.571844,5.880659,0,0,0,1
1082d 13h 55m 18s 813242µs,70179017,"""nfmcg_15432422""","""u2i""","""click""","""android""",1082,34.582149,3.571844,5.880659,0,0,0,1
1082d 13h 55m 28s 568179µs,50108628,"""nfmcg_19104642""","""catalog""","""click""","""android""",1082,34.582149,3.571844,5.880659,0,0,0,1
1082d 13h 55m 30s 111505µs,74607550,"""nfmcg_13704758""","""u2i""","""click""","""ios""",1082,34.582149,3.571844,5.880659,0,0,0,1
1082d 13h 55m 31s 19362µs,35174504,"""nfmcg_22999806""","""catalog""","""click""","""android""",1082,34.582149,3.571844,5.880659,0,0,0,1


## Global Temporal Split

In [10]:
def global_temporal_split(
    df: pl.LazyFrame, test_size: int | float = 1, date_column: str = "day"
) -> tuple[pl.LazyFrame, pl.LazyFrame]:
    """
    Разделяет датасет на обучающую и тестовую части на основе глобальной временной границы. 1 день между тестовой и обучающей частью игнорируется.

    Args:
        df: Датасет для разделения
        date_column: Имя столбца с датами
        test_size: Количество дней в тестовой части или доля от общего количества дней

    Returns:
        Кортеж из двух датасетов: обучающая и тестовая части
    """
    min_day, max_day = (
        df.select(
            pl.col(date_column).min().alias("min_day"), pl.col(date_column).max().alias("max_day")
        )
        .collect(engine="streaming")
        .row(0)
    )
    days_all = (max_day - min_day) + 1
    if isinstance(test_size, float):
        test_size = int(days_all * test_size)
        if test_size == 0:
            test_size += 1
        cut_day = max_day - test_size
    else:
        cut_day = max_day - test_size

    if cut_day - 1 < min_day or cut_day + 1 > max_day:
        raise ValueError(
            f"Test size is too large. Test size: {test_size}, min day: {min_day}, max day: {max_day}, cut day: {cut_day}"
        )

    train_df = df.filter(pl.col(date_column) < cut_day)
    test_df = df.filter(pl.col(date_column) > cut_day)

    return train_df, test_df
    

In [11]:
events_df.collect_schema()

Schema([('timestamp', Duration(time_unit='us')),
        ('user_id', UInt64),
        ('item_id', String),
        ('subdomain', String),
        ('action_type', String),
        ('os', String),
        ('day', Int32),
        ('target', Float64),
        ('log_target', Float64),
        ('sqrt_target', Float64),
        ('target_view', Int8),
        ('target_clickout', Int8),
        ('target_like', Int8),
        ('target_click', Int8)])

In [12]:
def get_last_k_user_interactions(
    events_df: pl.LazyFrame,
    last_k: int | None = 30,
    date_column: str = "day",
    timestamp_column: str = "timestamp",
    user_column: str = "user_id",
    acceptable_action: list[str] | None = None,
):
    if acceptable_action is None:
        acceptable_action = ["view", "clickout", "like", "click"]
    return (
        events_df.filter(pl.col("action_type").is_in(set(acceptable_action)))
        .group_by(user_column)
        .agg(
            pl.all().sort_by(date_column, timestamp_column).tail(last_k)
            if last_k is not None
            else pl.all().sort_by(date_column, timestamp_column)
        )
    )

In [13]:
events_df.collect_schema()

Schema([('timestamp', Duration(time_unit='us')),
        ('user_id', UInt64),
        ('item_id', String),
        ('subdomain', String),
        ('action_type', String),
        ('os', String),
        ('day', Int32),
        ('target', Float64),
        ('log_target', Float64),
        ('sqrt_target', Float64),
        ('target_view', Int8),
        ('target_clickout', Int8),
        ('target_like', Int8),
        ('target_click', Int8)])

In [14]:
train, test = global_temporal_split(events_df, 0.2)

In [15]:
get_last_k_user_interactions(test, acceptable_action=["click", "like", "clickout"]).collect(engine="streaming")

user_id,timestamp,item_id,subdomain,action_type,os,day,target,log_target,sqrt_target,target_view,target_clickout,target_like,target_click
u64,list[duration[μs]],list[str],list[str],list[str],list[str],list[i32],list[f64],list[f64],list[f64],list[i8],list[i8],list[i8],list[i8]
37527666,[1264d 16m 41s 497227µs],"[""nfmcg_25928977""]","[""u2i""]","[""click""]","[""android""]",[1264],[34.582149],[3.571844],[5.880659],[0],[0],[0],[1]
38168521,"[1306d 11h 15m 51s 408344µs, 1306d 12h 29m 23s 113283µs, … 1306d 20h 28m 8s 514160µs]","[""nfmcg_6755174"", ""nfmcg_6195336"", … ""nfmcg_6195336""]","[""catalog"", ""search"", … ""search""]","[""click"", ""clickout"", … ""click""]","[""other"", ""other"", … ""other""]","[1306, 1306, … 1306]","[34.582149, 268.714913, … 34.582149]","[3.571844, 5.597366, … 3.571844]","[5.880659, 16.392526, … 5.880659]","[0, 0, … 0]","[0, 1, … 0]","[0, 0, … 0]","[1, 0, … 1]"
41052669,"[1265d 13h 12m 41s 314192µs, 1265d 14h 34m 14s 291226µs]","[""nfmcg_2303161"", ""nfmcg_37761""]","[""u2i"", ""u2i""]","[""click"", ""click""]","[""android"", ""ios""]","[1265, 1265]","[34.582149, 34.582149]","[3.571844, 3.571844]","[5.880659, 5.880659]","[0, 0]","[0, 0]","[0, 0]","[1, 1]"
62975915,"[1278d 3h 5m 5s 789555µs, 1278d 8h 12m 20s 953620µs, … 1298d 19h 15m 46s 596112µs]","[""nfmcg_18850298"", ""nfmcg_21358372"", … ""nfmcg_4104717""]","[""u2i"", ""u2i"", … ""u2i""]","[""click"", ""click"", … ""click""]","[""android"", ""android"", … ""android""]","[1278, 1278, … 1298]","[34.582149, 34.582149, … 34.582149]","[3.571844, 3.571844, … 3.571844]","[5.880659, 5.880659, … 5.880659]","[0, 0, … 0]","[0, 0, … 0]","[0, 0, … 0]","[1, 1, … 1]"
6134291,"[1266d 21h 43m 46s 406104µs, 1267d 1h 54m 15s 682590µs, … 1300d 19h 30m 50s 417785µs]","[""nfmcg_11650133"", ""nfmcg_21700086"", … ""nfmcg_19079348""]","[""u2i"", ""u2i"", … ""u2i""]","[""click"", ""click"", … ""click""]","[""android"", ""ios"", … ""ios""]","[1266, 1267, … 1300]","[34.582149, 34.582149, … 34.582149]","[3.571844, 3.571844, … 3.571844]","[5.880659, 5.880659, … 5.880659]","[0, 0, … 0]","[0, 0, … 0]","[0, 0, … 0]","[1, 1, … 1]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…
43117819,[1275d 17h 27m 25s 230203µs],"[""nfmcg_2752201""]","[""u2i""]","[""click""]","[""android""]",[1275],[34.582149],[3.571844],[5.880659],[0],[0],[0],[1]
13718029,"[1268d 1h 16m 53s 747649µs, 1268d 6h 39m 56s 969676µs, … 1268d 16h 9m 19s 648661µs]","[""nfmcg_37761"", ""nfmcg_22889555"", … ""nfmcg_22889555""]","[""search"", ""search"", … ""search""]","[""click"", ""clickout"", … ""click""]","[""ios"", ""other"", … ""other""]","[1268, 1268, … 1268]","[34.582149, 268.714913, … 34.582149]","[3.571844, 5.597366, … 3.571844]","[5.880659, 16.392526, … 5.880659]","[0, 0, … 0]","[0, 1, … 0]","[0, 0, … 0]","[1, 0, … 1]"
58468842,"[1268d 6h 7m 50s 352027µs, 1268d 6h 33m 47s 339234µs, … 1273d 17h 38m 11s 339668µs]","[""nfmcg_22937488"", ""nfmcg_9173749"", … ""nfmcg_14924981""]","[""u2i"", ""i2i"", … ""search""]","[""click"", ""click"", … ""click""]","[""android"", ""ios"", … ""android""]","[1268, 1268, … 1273]","[34.582149, 34.582149, … 34.582149]","[3.571844, 3.571844, … 3.571844]","[5.880659, 5.880659, … 5.880659]","[0, 0, … 0]","[0, 0, … 0]","[0, 0, … 0]","[1, 1, … 1]"


In [ ]:
def split_cold_start(train_df: pl.LazyFrame, test_df: pl.LazyFrame, user_col: str = "user_id"):
    """
    Split test data into cold-start and non-cold-start subsets by users.

    Parameters
    ----------
    train_df : pl.LazyFrame
        Training data. Used to determine which users are already known to the model.
    test_df : pl.LazyFrame
        Test data that will be split into cold-start and non-cold-start parts.
    user_col : str, optional
        Name of the column containing user identifiers, by default "user_id".

    Returns
    -------
    tuple[pl.LazyFrame, pl.LazyFrame]
        A tuple of two LazyFrames:

        - first element : test subset with cold-start users
          (users present in `test_df` but not in `train_df`);
        - second element : test subset with non-cold-start users
          (users present both in `train_df` and `test_df`).
    """
    cold_start_users = test_df.select(pl.col(user_col).unique()).join(
        train_df.select(pl.col(user_col).unique()), on=user_col, how="anti"
    )
    return test_df.join(cold_start_users, on=user_col), test_df.join(
        cold_start_users, on=user_col, how="anti"
    )

In [16]:
def ndcg_at_k(
    user_based_df: pl.DataFrame,
    relevancy_col: str,
    true_items_col: str,
    predicted_items_col: str,
    predicted_score_col: str,
    top_k: int = 15,
):
    """
    Computes user-based NDCG@k for graded relevance in a recommendation setting.

    Parameters
    ----------
    user_based_df : pl.DataFrame
        Dataframe with user data. Each row must contain user and its lists with: truth
        ground items, their relevancy estimation and model prediction score.
    relevancy_col : str
        Column name contains list of relevancy estimations ground
        truth items (pl.List[float]) for user. Elements order must match `true_items_col`.
    true_items_col : str
        Column name of ground truth items with which user had interactions (pl.List[str]). Relevancy
        of these items must be set in `relevancy_col` respectively. 
    predicted_items_col : str
        Columns name with predicted items (pl.List[str]). Must be set in order matches
        `predicted_score_col`.
    predicted_score_col : str
        Columns name with predicted scores for items in `predicted_items_col` (pl.List[float]).
        Used to sort predictions in descending order.
    top_k : int, optional
        Top k elements to calculate `@k` metric.

    Returns
    -------
    pl.DataFrame
        Columns:
        - ``user_id`` : user identifier;
        - ``ndcg`` : NDCG@k for current user.

    Notes
    -----
    For each user, the function:
    1. Aggregates relevancies for ground-truth items by taking the maximum value for each item.
    2. Joins predicted items with their ground-truth relevancies.
    3. Computes DCG@k using the order induced by the model (sorting by score).
    4. Computes IDCG@k using the ideal order (sorting by ground-truth relevancy).
    5. Returns NDCG@k = DCG@k / IDCG@k, or 0.0 if IDCG@k = 0.
    """
    user_ids = []
    ndcgs = []
    for row in user_based_df.iter_rows(named=True):
        true_items = pl.DataFrame(
            {"truth_items": row[true_items_col], "relevancy": row[relevancy_col]}
        )
        true_items = true_items.group_by("truth_items").agg(
            pl.col("relevancy").max()
        )  # Берем максимальную релевантность для товара
        predictions = (
            pl.DataFrame(
                {"predicted_items": row[predicted_items_col], "score": row[predicted_score_col]}
            )
            .join(
                true_items,
                left_on="predicted_items",
                right_on="truth_items",
                coalesce=True,
                how="left",
            )
            .fill_null(0)
        )
        idcg = (
            predictions.select("relevancy")
            .sort("relevancy", descending=True)
            .head(top_k)
            .select((pl.col("relevancy") / (pl.row_index() + 2).log(2)).sum())
            .item()
        )
        dcg = (
            predictions.select("score", "relevancy")
            .sort("score", descending=True)
            .head(top_k)
            .select((pl.col("relevancy") / (pl.row_index() + 2).log(2)).sum())
            .item()
        )
        user_ids.append(row["user_id"])
        ndcgs.append(0.0 if idcg == 0 else dcg / idcg)
    return pl.DataFrame({"user_id": user_ids, "ndcg": ndcgs})

In [17]:
user_based_df = pl.DataFrame({
    "user_id": ["u1", "u2"],
    "true_items": [
        ["A", "B", "C"],   # истиные товары
        ["A", "B", "C"],
    ],
    "relevancy": [
        [3.0, 2.0, 1.0],   # u1: A=3, B=2, C=1
        [3.0, 2.0, 1.0],   # u2: A=3, B=2, C=1
    ],
    "predicted_items": [
        ["A", "B", "C"],   # u1: идеальное ранжирование
        ["C", "B", "A"],   # u2: худшее (перевёрнутое)
    ],
    "predicted_scores": [
        [0.9, 0.8, 0.7],   # по убыванию A>B>C
        [0.9, 0.8, 0.7],   # по убыванию C>B>A
    ],
})

In [18]:
ndcg_at_k(
    user_based_df,
    relevancy_col="relevancy",
    true_items_col="true_items",
    predicted_items_col="predicted_items",
    predicted_score_col="predicted_scores",
    top_k=3,
)

user_id,ndcg
str,f64
"""u1""",1.0
"""u2""",0.789998
